# FINAL RQ2 RESULTS — Reverse Symptom Inference

**Purpose:** Answer RQ2 by using the final Bayesian Network to infer marginal and multi-evidence posteriors, evaluate them only against held-out test patients, assess joint symptom profiles, and quantify uncertainty with bootstrap confidence intervals.

This notebook is part of the final, unified MSc research pipeline described in the research proposal. It is designed to save intermediate artifacts to Google Drive so that long-running computations can resume after a Colab restart.

**Important:** the old processed splits and old model artifacts are not treated as final because the supervisor raised a patient-level split/data-leakage concern. The raw dataset may be reused, but the final pipeline rebuilds the split and downstream models from that raw data.


In [2]:
# =========================
# FINAL PROJECT CONFIGURATION
# =========================
from google.colab import drive
drive.mount("/content/drive")

import os, json, pickle, shutil, hashlib, time, warnings
from pathlib import Path

PIPELINE_VERSION = "FINAL_2026-09-24_v1"
RANDOM_STATE = 42

DRIVE_ROOT = Path("/content/drive/MyDrive/heart_failure_bn_FINAL")
DIRS = {
    "root": DRIVE_ROOT,
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "models": DRIVE_ROOT / "models",
    "predictions": DRIVE_ROOT / "predictions",
    "results": DRIVE_ROOT / "results",
    "figures": DRIVE_ROOT / "figures",
    "logs": DRIVE_ROOT / "logs",
    "data": DRIVE_ROOT / "data",
    "templates": DRIVE_ROOT / "templates",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

LEGACY_ROOT = Path("/content/drive/MyDrive/heart_failure_bn_checkpoints")
print("FINAL Drive folder:", DRIVE_ROOT)
print("Pipeline version:", PIPELINE_VERSION)

# We may reuse the raw dataset only. Old processed splits/models are deliberately NOT reused.
legacy_raw = LEGACY_ROOT / "raw_master_dataset.pkl"
final_raw = DIRS["data"] / "raw_master_dataset.pkl"
if not final_raw.exists() and legacy_raw.exists():
    shutil.copy2(legacy_raw, final_raw)
    print("Copied legacy raw dataset into FINAL project folder.")
print("Raw dataset available:", final_raw.exists())


Mounted at /content/drive
FINAL Drive folder: /content/drive/MyDrive/heart_failure_bn_FINAL
Pipeline version: FINAL_2026-09-24_v1
Raw dataset available: True


## Research alignment

The proposal defines RQ2 as reverse inference from a confirmed diagnosis to marginal clinical-variable posteriors and top-K symptom profiles. Prof. Ajoodha additionally requires the evaluation to use held-out test data and to demonstrate inference that combines multiple pieces of clinical evidence.

This notebook therefore contains four layers:
1. Marginal inference given HF.
2. Multi-evidence conditional inference.
3. Joint top-K profile inference.
4. Held-out evaluation with MAD/Jaccard, log-likelihood and bootstrap uncertainty.

In [3]:
!pip install -q pgmpy

import os, json, pickle, math, gc
from collections import Counter
import numpy as np
import pandas as pd
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling
try:
    from pgmpy.factors.discrete import State
except ImportError:
    from pgmpy.utils import State
from sklearn.metrics import cohen_kappa_score

np.random.seed(RANDOM_STATE)

TARGET = "heart_failure"
N_SAMPLES = 20000
RQ2_BOOTSTRAPS = 1000
MIN_SCENARIO_CASES = 20

model_path = DIRS["models"] / "FINAL_bayesian_network.pkl"
assert model_path.exists(), "Run FINAL_RQ1_RESULTS.ipynb through BN training first."

with open(model_path, "rb") as f:
    consensus_model = pickle.load(f)

X_train_bn = pd.read_pickle(DIRS["data"] / "X_train_bn.pkl")
X_val_bn = pd.read_pickle(DIRS["data"] / "X_val_bn.pkl")
X_test_bn = pd.read_pickle(DIRS["data"] / "X_test_bn.pkl")

for df_ in [X_train_bn, X_val_bn, X_test_bn]:
    for c in df_.columns:
        df_[c] = df_[c].astype(str)

ve = VariableElimination(consensus_model)
variables = [v for v in consensus_model.nodes() if v != TARGET]

print("BN nodes:", len(consensus_model.nodes()))
print("BN edges:", len(consensus_model.edges()))
print("Test patients:", len(X_test_bn))
print("HF-positive test patients:", int((X_test_bn[TARGET] == "1").sum()))


BN nodes: 57
BN edges: 109
Test patients: 14096
HF-positive test patients: 3889


## 1. Marginal posteriors P(Sᵢ | HF=1) — test-independent model inference


In [4]:
marginal_path = DIRS["results"] / "FINAL_RQ2_marginal_posteriors.csv"

if marginal_path.exists():
    marginal_df = pd.read_csv(marginal_path)
else:
    rows = []
    for var in variables:
        q = ve.query([var], evidence={TARGET: "1"}, show_progress=False)
        for state, prob in zip(q.state_names[var], q.values):
            rows.append({
                "variable": var,
                "state": str(state),
                "posterior_given_HF1": float(prob)
            })
    marginal_df = pd.DataFrame(rows)
    marginal_df.to_csv(marginal_path, index=False)

print(marginal_df.to_string(max_rows=100))


                        variable           state  posterior_given_HF1
0                heart_rate_mean     bradycardia             0.037968
1                heart_rate_mean          normal             0.795492
2                heart_rate_mean     tachycardia             0.166539
3                 heart_rate_min     bradycardia             0.266372
4                 heart_rate_min          normal             0.699120
5                 heart_rate_min     tachycardia             0.034508
6              diastolic_bp_mean            high             0.107227
7              diastolic_bp_mean             low             0.314131
8              diastolic_bp_mean          normal             0.578642
9               diastolic_bp_min            high             0.019334
10              diastolic_bp_min             low             0.815572
11              diastolic_bp_min          normal             0.165094
12                           age        65_to_80             0.375823
13                  

## 2. Held-out observed-frequency comparison — TEST SET ONLY

In [5]:
# Critical correction: observed reference frequencies are computed only from held-out X_test_bn.
hf_test = X_test_bn[X_test_bn[TARGET] == "1"].copy()
assert len(hf_test) > 0

observed_rows = []
for var in variables:
    freqs = hf_test[var].value_counts(normalize=True)
    for state, freq in freqs.items():
        observed_rows.append({
            "variable": var,
            "state": str(state),
            "observed_freq_test_HF1": float(freq)
        })

observed_df = pd.DataFrame(observed_rows)

alignment_df = marginal_df.merge(
    observed_df, on=["variable", "state"], how="outer"
)
alignment_df["posterior_given_HF1"] = alignment_df["posterior_given_HF1"].fillna(0.0)
alignment_df["observed_freq_test_HF1"] = alignment_df["observed_freq_test_HF1"].fillna(0.0)
alignment_df["abs_diff"] = (
    alignment_df["posterior_given_HF1"] - alignment_df["observed_freq_test_HF1"]
).abs()

mad_per_variable = (
    alignment_df.groupby("variable")["abs_diff"]
    .mean()
    .sort_values(ascending=False)
)
overall_mad = float(alignment_df["abs_diff"].mean())

alignment_df.to_csv(DIRS["results"] / "FINAL_RQ2_marginal_vs_test_observed.csv", index=False)
mad_per_variable.to_csv(DIRS["results"] / "FINAL_RQ2_MAD_per_variable.csv")

print("Overall MAD:", overall_mad)
print(mad_per_variable)


Overall MAD: 0.020659967644301946
variable
mean_arterial_pressure_min     0.067849
systolic_bp_min                0.060461
diastolic_bp_mean              0.050629
age                            0.049485
diastolic_bp_min               0.048554
respiratory_rate_min           0.046989
diabetes                       0.044868
mean_arterial_pressure_mean    0.044537
haemoglobin_max                0.043942
haemoglobin_mean               0.038300
glucose_max                    0.037788
diastolic_bp_max               0.035590
haemoglobin_min                0.032037
lactate_max                    0.027091
glucose_mean                   0.026181
lactate_mean                   0.023213
heart_rate_max                 0.022152
systolic_bp_max                0.021296
sodium_min                     0.020443
temperature_min                0.020127
sodium_mean                    0.019316
temperature_max                0.018640
respiratory_rate_mean          0.018452
gender                         0.0177

## 3. Held-out log-likelihood of the joint distribution

In [6]:
def row_log_probability(model, row):
    total_log = 0.0
    for node in model.nodes():
        cpd = model.get_cpds(node)
        assignment = {node: str(row[node])}
        for parent in model.get_parents(node):
            assignment[parent] = str(row[parent])
        try:
            p = float(cpd.get_value(**assignment))
        except Exception:
            # Safer fallback: explicit factor-index lookup through state names.
            p = 0.0
            try:
                node_states = [str(s) for s in cpd.state_names[node]]
                node_idx = node_states.index(str(row[node]))
                parent_indices = []
                for parent in cpd.variables[1:]:
                    states = [str(s) for s in cpd.state_names[parent]]
                    parent_indices.append(states.index(str(row[parent])))
                idx = (node_idx, *parent_indices)
                p = float(cpd.values[idx])
            except Exception as e:
                raise RuntimeError(f"Could not score node={node}: {e}")
        total_log += math.log(max(p, 1e-300))
    return total_log

ll_values = []
for _, row in X_test_bn.iterrows():
    ll_values.append(row_log_probability(consensus_model, row))

ll_values = np.asarray(ll_values)
joint_ll = pd.DataFrame({
    "test_row_log_likelihood": ll_values
})
joint_ll.to_csv(DIRS["results"] / "FINAL_RQ2_heldout_log_likelihood_per_patient.csv", index=False)

print("Held-out mean log-likelihood per patient:", float(ll_values.mean()))
print("Held-out total log-likelihood:", float(ll_values.sum()))


Held-out mean log-likelihood per patient: -25.000346192868886
Held-out total log-likelihood: -352404.87993467983


## 4. Joint top-K profiles via likelihood-weighted sampling

In [7]:
# The profile probabilities are weighted joint probabilities from the BN sampler;
# they are NOT products of independent marginal probabilities.
profile_path = DIRS["results"] / "FINAL_RQ2_top20_profiles.csv"

if profile_path.exists():
    top_profiles_df = pd.read_csv(profile_path)
else:
    np.random.seed(RANDOM_STATE)
    sampler = BayesianModelSampling(consensus_model)
    samples = sampler.likelihood_weighted_sample(
        evidence=[State(TARGET, "1")],
        size=N_SAMPLES,
        show_progress=False
    )
    samples = samples.astype(str)

    weight_col = "_weight"
    profile_cols = variables
    keys = samples[profile_cols].astype(str).agg("|".join, axis=1)

    grouped = (
        pd.DataFrame({"key": keys, "weight": samples[weight_col].astype(float)})
        .groupby("key")["weight"].sum()
        .sort_values(ascending=False)
    )
    probs = grouped / grouped.sum()

    rows = []
    for rank, (key, prob) in enumerate(probs.head(20).items(), start=1):
        vals = key.split("|")
        row = {"rank": rank, "joint_probability_estimate": float(prob), "method": "likelihood_weighted_sampling"}
        row.update(dict(zip(profile_cols, vals)))
        rows.append(row)

    top_profiles_df = pd.DataFrame(rows)
    top_profiles_df.to_csv(profile_path, index=False)

    pd.DataFrame({
        "n_states_product": [math.prod(
            [len(consensus_model.get_cpds(v).state_names[v]) for v in variables]
        )]
    }).to_csv(DIRS["results"] / "FINAL_RQ2_configuration_space.csv", index=False)

print(top_profiles_df.head(20).to_string(index=False))

cum_mass = []
for k in [5, 10, 20]:
    mass = float(top_profiles_df.head(k)["joint_probability_estimate"].sum())
    cum_mass.append({"K": k, "cumulative_probability_mass": mass})
pd.DataFrame(cum_mass).to_csv(DIRS["results"] / "FINAL_RQ2_topK_cumulative_mass.csv", index=False)
print(pd.DataFrame(cum_mass))


 rank  joint_probability_estimate                       method heart_rate_mean heart_rate_min diastolic_bp_mean diastolic_bp_min      age heart_rate_max respiratory_rate_mean respiratory_rate_max   spo2_min temperature_max  spo2_mean spo2_max creatinine_mean creatinine_min  ckd   glucose_mean    glucose_max  diabetes haemoglobin_mean haemoglobin_max haemoglobin_min lactate_mean lactate_min temperature_mean platelet_count_mean platelet_count_min platelet_count_max sodium_mean sodium_max bun_mean bun_min white_blood_cells_mean white_blood_cells_min white_blood_cells_max  coronary_artery_disease  copd systolic_bp_min systolic_bp_mean systolic_bp_max mean_arterial_pressure_min mean_arterial_pressure_max diastolic_bp_max mean_arterial_pressure_mean temperature_min gender lactate_max potassium_max potassium_min sodium_min bun_max  hypertension  atrial_fibrillation creatinine_max    glucose_min potassium_mean respiratory_rate_min
    1                    0.000105 likelihood_weighted_sampling 

## 5. Profile similarity — multiple observed profiles + abnormal-state supplement

In [8]:
# State-aware Jaccard. Full-profile Jaccard is reported, but we supplement it with an
# abnormal/clinically-interesting-state version to avoid normal-state inflation.

def full_profile_set(row_dict, cols):
    return {(c, str(row_dict[c])) for c in cols}

def is_abnormal_state(variable, state):
    s = str(state).strip().lower()
    if variable in {"gender", "admission_type", "age"}:
        return False
    if variable in {"hypertension", "diabetes", "ckd", "copd", "coronary_artery_disease", "atrial_fibrillation"}:
        return s in {"1", "true", "yes", "present"}
    normal_tokens = {"normal", "absent", "no", "none", "false", "0"}
    return s not in normal_tokens and "under_65" not in s

def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 1.0

observed_keys = hf_test[variables].astype(str).agg("|".join, axis=1).value_counts()
observed_profiles = []
for key in observed_keys.head(10).index:
    observed_profiles.append(dict(zip(variables, key.split("|"))))

profile_rows = []
for _, inferred in top_profiles_df.iterrows():
    inferred_dict = inferred.to_dict()
    a_full = full_profile_set(inferred_dict, variables)
    a_abn = {(c, str(inferred_dict[c])) for c in variables if is_abnormal_state(c, inferred_dict[c])}

    full_scores = []
    abn_scores = []
    for obs in observed_profiles:
        b_full = full_profile_set(obs, variables)
        b_abn = {(c, str(obs[c])) for c in variables if is_abnormal_state(c, obs[c])}
        full_scores.append(jaccard(a_full, b_full))
        abn_scores.append(jaccard(a_abn, b_abn))

    profile_rows.append({
        "rank": inferred["rank"],
        "joint_probability_estimate": inferred["joint_probability_estimate"],
        "max_jaccard_vs_top10_observed": max(full_scores),
        "mean_jaccard_vs_top10_observed": float(np.mean(full_scores)),
        "max_abnormal_state_jaccard_vs_top10_observed": max(abn_scores),
        "mean_abnormal_state_jaccard_vs_top10_observed": float(np.mean(abn_scores)),
    })

profile_similarity_df = pd.DataFrame(profile_rows)
profile_similarity_df.to_csv(DIRS["results"] / "FINAL_RQ2_profile_similarity.csv", index=False)
print(profile_similarity_df.to_string(index=False))


 rank  joint_probability_estimate  max_jaccard_vs_top10_observed  mean_jaccard_vs_top10_observed  max_abnormal_state_jaccard_vs_top10_observed  mean_abnormal_state_jaccard_vs_top10_observed
    1                    0.000105                       0.365854                        0.309658                                      0.372093                                       0.239147
    2                    0.000105                       0.473684                        0.388773                                      0.368421                                       0.258617
    3                    0.000105                       0.623188                        0.443692                                      0.533333                                       0.351153
    4                    0.000105                       0.534247                        0.393369                                      0.428571                                       0.271340
    5                    0.000105                 

## 6. Bootstrap confidence intervals for RQ2

In [9]:
# Bootstrap the HELD-OUT test HF patients. The BN posterior is fixed; the observed
# empirical distribution is resampled. This quantifies uncertainty in the comparison.

rng = np.random.RandomState(RANDOM_STATE)
hf_test_reset = hf_test.reset_index(drop=True)

bootstrap_mad = []
bootstrap_jaccard = {rank: [] for rank in top_profiles_df["rank"].tolist()}
bootstrap_abn_jaccard = {rank: [] for rank in top_profiles_df["rank"].tolist()}

for b in range(RQ2_BOOTSTRAPS):
    sample_idx = rng.randint(0, len(hf_test_reset), len(hf_test_reset))
    sample = hf_test_reset.iloc[sample_idx]

    abs_diffs = []
    for _, row in marginal_df.iterrows():
        var, state = row["variable"], str(row["state"])
        freq = float((sample[var].astype(str) == state).mean())
        abs_diffs.append(abs(float(row["posterior_given_HF1"]) - freq))
    bootstrap_mad.append(float(np.mean(abs_diffs)))

    observed_keys_b = sample[variables].astype(str).agg("|".join, axis=1).value_counts()
    obs_b = [dict(zip(variables, key.split("|"))) for key in observed_keys_b.head(10).index]

    for _, inferred in top_profiles_df.iterrows():
        rank = int(inferred["rank"])
        ia = {(c, str(inferred[c])) for c in variables}
        ia_abn = {(c, str(inferred[c])) for c in variables if is_abnormal_state(c, inferred[c])}
        fulls, abns = [], []
        for obs in obs_b:
            oa = {(c, str(obs[c])) for c in variables}
            oa_abn = {(c, str(obs[c])) for c in variables if is_abnormal_state(c, obs[c])}
            fulls.append(jaccard(ia, oa))
            abns.append(jaccard(ia_abn, oa_abn))
        bootstrap_jaccard[rank].append(max(fulls) if fulls else np.nan)
        bootstrap_abn_jaccard[rank].append(max(abns) if abns else np.nan)

mad_ci = {
    "metric": "overall_MAD",
    "estimate": overall_mad,
    "ci_low": float(np.percentile(bootstrap_mad, 2.5)),
    "ci_high": float(np.percentile(bootstrap_mad, 97.5))
}

ci_rows = [mad_ci]
for rank in bootstrap_jaccard:
    point = float(profile_similarity_df.loc[profile_similarity_df["rank"] == rank, "max_jaccard_vs_top10_observed"].iloc[0])
    ci_rows.append({
        "metric": f"rank{rank}_max_jaccard_top10",
        "estimate": point,
        "ci_low": float(np.nanpercentile(bootstrap_jaccard[rank], 2.5)),
        "ci_high": float(np.nanpercentile(bootstrap_jaccard[rank], 97.5)),
    })
    abn_point = float(profile_similarity_df.loc[profile_similarity_df["rank"] == rank, "max_abnormal_state_jaccard_vs_top10_observed"].iloc[0])
    ci_rows.append({
        "metric": f"rank{rank}_max_abnormal_jaccard_top10",
        "estimate": abn_point,
        "ci_low": float(np.nanpercentile(bootstrap_abn_jaccard[rank], 2.5)),
        "ci_high": float(np.nanpercentile(bootstrap_abn_jaccard[rank], 97.5)),
    })

rq2_bootstrap_summary = pd.DataFrame(ci_rows)
rq2_bootstrap_summary.to_csv(DIRS["results"] / "FINAL_RQ2_bootstrap_CIs.csv", index=False)
print(rq2_bootstrap_summary.to_string(index=False))


                           metric  estimate   ci_low  ci_high
                      overall_MAD  0.020660 0.019141 0.023355
          rank1_max_jaccard_top10  0.365854 0.365854 0.555556
 rank1_max_abnormal_jaccard_top10  0.372093 0.297251 0.514286
          rank2_max_jaccard_top10  0.473684 0.435897 0.671642
 rank2_max_abnormal_jaccard_top10  0.368421 0.322552 0.592593
          rank3_max_jaccard_top10  0.623188 0.435897 0.696970
 rank3_max_abnormal_jaccard_top10  0.533333 0.363573 0.645161
          rank4_max_jaccard_top10  0.534247 0.454545 0.723077
 rank4_max_abnormal_jaccard_top10  0.428571 0.333333 0.652174
          rank5_max_jaccard_top10  0.417722 0.400000 0.623188
 rank5_max_abnormal_jaccard_top10  0.439024 0.342838 0.593750
          rank6_max_jaccard_top10  0.473684 0.417278 0.600000
 rank6_max_abnormal_jaccard_top10  0.439024 0.380888 0.625000
          rank7_max_jaccard_top10  0.534247 0.493333 0.696970
 rank7_max_abnormal_jaccard_top10  0.542857 0.473684 0.710526
        

## 7. Multi-evidence inference — the key supervisor-requested RQ2 analysis

In [10]:
# These are clinically interpretable evidence combinations. A scenario is used only
# if enough HELD-OUT test patients have that exact combination.
candidate_scenarios = [
    {"name": "HF_plus_tachycardia_plus_elevated_creatinine",
     "evidence": {"heart_rate_mean": "tachycardia", "creatinine_mean": "elevated"}},
    {"name": "HF_plus_hypoxaemia_plus_tachycardia",
     "evidence": {"spo2_mean": "hypoxaemic", "heart_rate_mean": "tachycardia"}},
    {"name": "HF_plus_hypotension_plus_elevated_lactate",
     "evidence": {"systolic_bp_mean": "hypotensive", "lactate_mean": "elevated"}},
    {"name": "HF_plus_hypoxaemia_plus_high_creatinine",
     "evidence": {"spo2_mean": "hypoxaemic", "creatinine_mean": "high"}},
]

scenario_rows = []

for scenario in candidate_scenarios:
    evidence = {TARGET: "1", **scenario["evidence"]}
    required = [k for k in evidence if k in X_test_bn.columns]
    if len(required) != len(evidence):
        continue

    mask = np.ones(len(X_test_bn), dtype=bool)
    for k, v in evidence.items():
        mask &= X_test_bn[k].astype(str).to_numpy() == str(v)
    matched = X_test_bn.loc[mask]

    if len(matched) < MIN_SCENARIO_CASES:
        print("Skipping", scenario["name"], "— only", len(matched), "held-out cases.")
        continue

    remaining = [v for v in variables if v not in scenario["evidence"]]
    rows = []
    for var in remaining:
        q = ve.query([var], evidence=evidence, show_progress=False)
        post = {str(s): float(p) for s, p in zip(q.state_names[var], q.values)}
        obs = matched[var].astype(str).value_counts(normalize=True).to_dict()
        states = sorted(set(post) | set(obs))
        mad = float(np.mean([abs(post.get(s,0.0)-obs.get(s,0.0)) for s in states]))
        rows.append({
            "scenario": scenario["name"],
            "n_test_cases": len(matched),
            "variable": var,
            "conditional_MAD": mad,
        })

    scenario_df = pd.DataFrame(rows)
    scenario_df.to_csv(DIRS["results"] / f"FINAL_RQ2_{scenario['name']}.csv", index=False)
    scenario_rows.append({
        "scenario": scenario["name"],
        "n_test_cases": len(matched),
        "mean_conditional_MAD": scenario_df["conditional_MAD"].mean()
    })

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary.to_csv(DIRS["results"] / "FINAL_RQ2_multi_evidence_summary.csv", index=False)
print(scenario_summary.to_string(index=False))


                                    scenario  n_test_cases  mean_conditional_MAD
HF_plus_tachycardia_plus_elevated_creatinine           182              0.047518
         HF_plus_hypoxaemia_plus_tachycardia            21              0.104435
   HF_plus_hypotension_plus_elevated_lactate            38              0.074617
     HF_plus_hypoxaemia_plus_high_creatinine            28              0.126192


## 8. Clinical plausibility assessment template

In [11]:
# Creates the structured rater template specified in the proposal.
rating_cols = [
    "profile_rank",
    "coherence_rater1", "completeness_rater1", "recognisability_rater1",
    "coherence_rater2", "completeness_rater2", "recognisability_rater2",
    "rater1_comments", "rater2_comments"
]
template = top_profiles_df[["rank"] + variables].copy()
template = template.rename(columns={"rank": "profile_rank"})
for c in rating_cols:
    if c not in template.columns:
        template[c] = ""
template = template[rating_cols]
template.to_csv(DIRS["templates"] / "FINAL_RQ2_clinical_rater_template.csv", index=False)
print("Template:", DIRS["templates"] / "FINAL_RQ2_clinical_rater_template.csv")


Template: /content/drive/MyDrive/heart_failure_bn_FINAL/templates/FINAL_RQ2_clinical_rater_template.csv


## 9. Final RQ2 summary artifact

In [12]:

summary = pd.DataFrame([{
    "overall_MAD": overall_mad,
    "heldout_mean_log_likelihood": float(ll_values.mean()),
    "top5_probability_mass": float(top_profiles_df.head(5)["joint_probability_estimate"].sum()),
    "top10_probability_mass": float(top_profiles_df.head(10)["joint_probability_estimate"].sum()),
    "top20_probability_mass": float(top_profiles_df.head(20)["joint_probability_estimate"].sum()),
    "n_test_HF_patients": int(len(hf_test)),
    "n_multi_evidence_scenarios_evaluated": int(len(scenario_summary)),
}])
summary.to_csv(DIRS["results"] / "FINAL_RQ2_RESULTS.csv", index=False)
print(summary.to_string(index=False))

 overall_MAD  heldout_mean_log_likelihood  top5_probability_mass  top10_probability_mass  top20_probability_mass  n_test_HF_patients  n_multi_evidence_scenarios_evaluated
     0.02066                   -25.000346               0.000524                0.001048                0.002096                3889                                     4


In [13]:

import numpy as np
import pandas as pd
import math

# Use the existing RQ2 notebook variables:
# X_train_bn, X_test_bn, consensus_model, DIRS

alpha = 1.0  # Add-one smoothing
nodes = list(consensus_model.nodes())
baseline_log_likelihoods = []

# Estimate each variable's marginal distribution from training data only.
marginal_probabilities = {}

for node in nodes:
    train_values = X_train_bn[node].astype(str)

    cpd = consensus_model.get_cpds(node)
    states = [str(s) for s in cpd.state_names[node]]

    n_train = len(train_values)
    k_states = len(states)

    counts = train_values.value_counts().to_dict()

    marginal_probabilities[node] = {
        state: (
            counts.get(state, 0) + alpha
        ) / (
            n_train + alpha * k_states
        )
        for state in states
    }

# Score each held-out test record using the independent-marginals baseline.
for _, row in X_test_bn.iterrows():
    total_log_probability = 0.0

    for node in nodes:
        state = str(row[node])
        probabilities = marginal_probabilities[node]

        if state not in probabilities:
            raise ValueError(
                f"Unknown state {state!r} for {node}. "
                "Check train/test state consistency."
            )

        total_log_probability += math.log(
            probabilities[state]
        )

    baseline_log_likelihoods.append(total_log_probability)

# Save per-patient baseline results.
baseline_df = pd.DataFrame({
    "test_row_log_likelihood_marginals_only":
        baseline_log_likelihoods
})

baseline_path = (
    DIRS["results"] /
    "FINAL_RQ2_marginals_only_baseline_log_likelihood.csv"
)

baseline_df.to_csv(baseline_path, index=False)

print("Marginals-only mean log-likelihood:",
      baseline_df.iloc[:, 0].mean())

print("Marginals-only total log-likelihood:",
      baseline_df.iloc[:, 0].sum())

print("Saved to:", baseline_path)


Marginals-only mean log-likelihood: -37.711825544207215
Marginals-only total log-likelihood: -531585.8928711449
Saved to: /content/drive/MyDrive/heart_failure_bn_FINAL/results/FINAL_RQ2_marginals_only_baseline_log_likelihood.csv
